# Guía de Laboratorio N.° 3 — Automatización y Calidad de Código
### Tecnologías Emergentes (ISO46B) · Semana 3 (cierre de la Unidad I) · Facultad de Ingeniería de Sistemas — UNCP

**Versión: SOLUCIONARIO** — incluye salidas esperadas y verificaciones automáticas.

Continúa sobre el repositorio `tecemer-lab1` de las Semanas 1 y 2. Ejecute este notebook desde la raíz de ese repositorio. Los comandos de instalación usan `.venv/bin/...` (Linux/macOS, incluido Colab); en Windows, fuera del notebook, use `.venv\\Scripts\\...`. **Cuando una celda dice "agregar a organizador.py", el código se suma al mismo archivo, no crea uno nuevo.**

## Requisitos previos
- Repositorio `tecemer-lab1` de las Semanas 1 y 2.
- Haber completado los Laboratorios N.° 1 y N.° 2.

## Parte 1 — pathlib + shutil: organizador de archivos (60 min) · Nivel: Básico

### Paso 1.1 — Carpeta de prueba

In [ ]:
from pathlib import Path

carpeta_prueba = Path("carpeta_prueba")
carpeta_prueba.mkdir(exist_ok=True)
(carpeta_prueba / "foto.jpg").write_text("contenido")
(carpeta_prueba / "informe.pdf").write_text("contenido")
(carpeta_prueba / "video.mp4").write_text("contenido")
print("Carpeta de prueba lista.")

### Paso 1.2 — Núcleo de clasificación con pathlib

In [ ]:
%%writefile organizador.py
from pathlib import Path

CATEGORIAS = {
    ".pdf": "Documentos", ".docx": "Documentos", ".txt": "Documentos",
    ".jpg": "Imagenes", ".jpeg": "Imagenes", ".png": "Imagenes",
    ".mp4": "Videos", ".mov": "Videos",
    ".zip": "Comprimidos", ".rar": "Comprimidos",
}


def clasificar(archivo):
    return CATEGORIAS.get(archivo.suffix.lower(), "Otros")


### Paso 1.3 — Mover archivos con shutil (se agrega al final de organizador.py)

In [ ]:
%%writefile -a organizador.py
import shutil


def organizar_carpeta(carpeta, simulacion=False):
    movidos = 0
    for archivo in carpeta.iterdir():
        if archivo.is_dir() or archivo.name.startswith("."):
            continue
        categoria = clasificar(archivo)
        destino_carpeta = carpeta / categoria
        destino = destino_carpeta / archivo.name

        if simulacion:
            print(f"[SIMULACION] {archivo.name} -> {categoria}")
        else:
            destino_carpeta.mkdir(exist_ok=True)
            shutil.move(str(archivo), str(destino))
            print(f"Movido: {archivo.name} -> {categoria}")
        movidos += 1
    return movidos


### Paso 1.4 — Interfaz de línea de comandos con argparse (se agrega al final)

In [ ]:
%%writefile -a organizador.py
import argparse


def main():
    parser = argparse.ArgumentParser(description="Organiza archivos por tipo.")
    parser.add_argument("carpeta", type=Path, help="Carpeta a organizar")
    parser.add_argument("--dry-run", action="store_true", help="Solo simula, no mueve archivos")
    args = parser.parse_args()

    total = organizar_carpeta(args.carpeta, simulacion=args.dry_run)
    print(f"Total de archivos procesados: {total}")


if __name__ == "__main__":
    main()


In [ ]:
!.venv/bin/python organizador.py carpeta_prueba --dry-run

In [ ]:
!.venv/bin/python organizador.py carpeta_prueba

✅ **Salida esperada:** en `--dry-run`, tres líneas `[SIMULACION]` sin mover nada. Sin la bandera, se crean las subcarpetas `Imagenes`, `Documentos` y `Videos` dentro de `carpeta_prueba`.

**Checkpoint de Parte 1:** verifique con `!ls carpeta_prueba` que aparecen las subcarpetas.

In [ ]:
!ls carpeta_prueba

## Parte 2 — Pruebas unitarias con pytest y tmp_path (90 min) · Nivel: Intermedio

### Paso 2.1 — Instalar pytest

In [ ]:
!.venv/bin/pip install pytest
!.venv/bin/pip freeze > requirements.txt

### Paso 2.2 — Primera prueba: clasificación y movimiento

In [ ]:
%%writefile test_organizador.py
from organizador import clasificar, organizar_carpeta


def test_clasificar_reconoce_imagen(tmp_path):
    archivo = tmp_path / "foto.jpg"
    archivo.write_text("contenido")
    assert clasificar(archivo) == "Imagenes"


def test_organizar_mueve_archivo_a_subcarpeta(tmp_path):
    archivo = tmp_path / "foto.jpg"
    archivo.write_text("contenido")

    organizar_carpeta(tmp_path)

    assert (tmp_path / "Imagenes" / "foto.jpg").exists()
    assert not archivo.exists()


### Paso 2.3 — Segunda prueba: modo simulación no debe mover nada (se agrega al final)

In [ ]:
%%writefile -a test_organizador.py


def test_organizar_en_modo_simulacion_no_mueve_nada(tmp_path):
    archivo = tmp_path / "foto.jpg"
    archivo.write_text("contenido")

    organizar_carpeta(tmp_path, simulacion=True)

    assert archivo.exists()
    assert not (tmp_path / "Imagenes").exists()


### Paso 2.4 — Ejecutar pytest y leer el reporte

In [ ]:
!.venv/bin/pytest -v

✅ **Salida esperada:** `collected 3 items`, las tres pruebas en `PASSED`, resumen final `3 passed`.

### Paso 2.5 — Checkpoint: commit de Parte 2

In [ ]:
!git add organizador.py test_organizador.py requirements.txt
!git commit -m "feat: agrega organizador de archivos con pruebas pytest"

## Parte 3 — Documentación técnica y consolidación (90 min) · Nivel: Avanzado — Cierre de la Unidad I

### Paso 3.1 — Instalar docformatter

In [ ]:
!.venv/bin/pip install docformatter

### Paso 3.2 — Reescribir organizador.py con docstrings, estilo Google y type hints (reemplaza todo el archivo)

In [ ]:
%%writefile organizador.py
"""Organiza los archivos de una carpeta en subcarpetas segun su extension."""
import argparse
import shutil
from pathlib import Path

CATEGORIAS = {
    ".pdf": "Documentos", ".docx": "Documentos", ".txt": "Documentos",
    ".jpg": "Imagenes", ".jpeg": "Imagenes", ".png": "Imagenes",
    ".mp4": "Videos", ".mov": "Videos",
    ".zip": "Comprimidos", ".rar": "Comprimidos",
}


def clasificar(archivo: Path) -> str:
    """Determina la categoria de un archivo segun su extension.

    Args:
        archivo: Ruta del archivo a clasificar.

    Returns:
        Nombre de la categoria correspondiente, u "Otros" si la
        extension no esta registrada.
    """
    return CATEGORIAS.get(archivo.suffix.lower(), "Otros")


def organizar_carpeta(carpeta: Path, simulacion: bool = False) -> int:
    """Organiza los archivos de una carpeta en subcarpetas por categoria.

    Args:
        carpeta: Carpeta cuyos archivos se van a organizar.
        simulacion: Si es True, solo muestra que haria sin mover archivos.

    Returns:
        Numero de archivos movidos (o que se moverian, en modo simulacion).
    """
    movidos = 0
    for archivo in carpeta.iterdir():
        if archivo.is_dir() or archivo.name.startswith("."):
            continue
        categoria = clasificar(archivo)
        destino_carpeta = carpeta / categoria
        destino = destino_carpeta / archivo.name

        if simulacion:
            print(f"[SIMULACION] {archivo.name} -> {categoria}")
        else:
            destino_carpeta.mkdir(exist_ok=True)
            shutil.move(str(archivo), str(destino))
            print(f"Movido: {archivo.name} -> {categoria}")
        movidos += 1
    return movidos


def main() -> None:
    """Punto de entrada de linea de comandos del organizador."""
    parser = argparse.ArgumentParser(description="Organiza archivos por tipo.")
    parser.add_argument("carpeta", type=Path, help="Carpeta a organizar")
    parser.add_argument(
        "--dry-run", action="store_true", help="Solo simula, no mueve archivos"
    )
    args = parser.parse_args()

    total = organizar_carpeta(args.carpeta, simulacion=args.dry_run)
    print(f"Total de archivos procesados: {total}")


if __name__ == "__main__":
    main()


In [ ]:
!.venv/bin/pytest -v

✅ **Salida esperada:** las 3 pruebas siguen en `PASSED` — documentar no debe cambiar el comportamiento.

### Paso 3.3 — Formatear los docstrings con docformatter

In [ ]:
!.venv/bin/docformatter --in-place organizador.py

### Paso 3.4 — README de cierre de la Unidad I (se agrega al final de README.md)

In [ ]:
%%writefile -a README.md

## Cierre de la Unidad I - Semana 3

Herramienta de automatizacion: organizador.py clasifica y mueve archivos
de una carpeta en subcarpetas por tipo (Documentos, Imagenes, Videos,
Comprimidos, Otros), con modo de simulacion (--dry-run) mediante argparse.

Uso:
```
python organizador.py <carpeta> [--dry-run]
```

Pruebas: test_organizador.py cubre clasificacion, movimiento real y modo
simulacion, usando la fixture tmp_path de pytest para no afectar el
sistema de archivos real. Ejecutar con: pytest -v


### Paso 3.5 — Commit final y cierre de la Unidad I

In [ ]:
!git add organizador.py README.md
!git commit -m "docs: documenta organizador con estilo Google y cierra la Unidad I"
!git push origin main

## Entregable de la sesión
El mismo repositorio de GitHub de las Semanas 1 y 2, con `organizador.py` (documentado en estilo Google, con type hints y formateado con docformatter) y `test_organizador.py` (mínimo 3 pruebas con `tmp_path`, todas en verde), el `README.md` con la sección de cierre de la Unidad I, y un commit final en formato Conventional Commits.

## Verificación automática (solo en el notebook del solucionario)

In [ ]:
from pathlib import Path

checks = {
    "organizador.py existe": Path("organizador.py").exists(),
    "test_organizador.py existe": Path("test_organizador.py").exists(),
}

for nombre, resultado in checks.items():
    estado = "\u2705" if resultado else "\u274c"
    print(f"{estado} {nombre}")

if Path("organizador.py").exists():
    contenido = Path("organizador.py").read_text()
    tiene_docstrings = '"""' in contenido
    tiene_type_hints = "-> str" in contenido and "-> int" in contenido
    print(f"{'\u2705' if tiene_docstrings else '\u274c'} organizador.py tiene docstrings")
    print(f"{'\u2705' if tiene_type_hints else '\u274c'} organizador.py usa type hints en clasificar y organizar_carpeta")

In [ ]:
!.venv/bin/pytest -v

In [ ]:
!git log --oneline -5

✅ **Salida esperada del historial:** el commit más reciente inicia con `docs:` y menciona el cierre de la Unidad I.

## Mapeo rápido a la rúbrica

| Criterio de la rúbrica | Evidencia en este notebook |
|---|---|
| Script de automatización | `--dry-run` no mueve archivos; modo real crea las subcarpetas correctas |
| Pruebas con pytest y tmp_path | `pytest -v` con 3 pruebas PASSED, todas usando `tmp_path` |
| Documentación técnica | Verificación automática de docstrings y type hints en `organizador.py` |
| Consolidación y cierre de Unidad I | `git log` muestra el commit `docs:` final; README con la sección de cierre |
